# Verify the `1shw` detvar store's cuts_signature drift check

Standalone, doesn't need `mcbnb_df`/the rest of `hnl_analysis_v6.ipynb` loaded --
only exercises `get_detvar_systs` directly against the freshly-built
`detvars_1shw.h5`. Run top to bottom; every code cell prints PASS/FAIL.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/exp/sbnd/data/users/lnguyen/cafpyana_pi0")

import numpy as np
import cafpybara.analyses.hnlpi0 as ana

## Load the store, confirm it's actually stamped

In [ ]:
store_path = ana.config.HNL_DETVAR_DIR + "/detvars_1shw.h5"

info = ana.detvar_store_info(store_path)
print(info)

assert 'cuts_signature' in info.columns, "FAIL: no cuts_signature column at all -- store predates the stamping code"
assert info['cuts_signature'].notna().all(), "FAIL: cuts_signature is present but empty/None for some/all groups"
print("\nPASS: every group has a non-null cuts_signature")

hnl_detvar_1shw = ana.load_detvar_dict(store_path)
first_group = next(iter(hnl_detvar_1shw))
print(f"\nExample stamp ('{first_group}'):", hnl_detvar_1shw[first_group]['cuts_signature'])

## Shared var/bins (matches `hnl_analysis_v6.ipynb`'s systematics cell)

In [ ]:
var = ('slc', 'barycenterFM', 'flashTime_calib_mod')
bins = np.linspace(0, 19, 20)

## Case (a): live cuts match the stamp exactly -- should run clean

In [ ]:
try:
    result = ana.get_detvar_systs(hnl_detvar_1shw, var, bins, cuts=ana.PI0_CUT_LISTS['1shw'])
    print("PASS: no-drift case ran clean, no error")
    print("Groups:", list(result.keys()))
except ValueError as e:
    print("FAIL: unexpected raise on the exact cuts that built this store:")
    print(e)

## Case (b): same cut name, drifted definition -- should raise

Finds a threshold-style cut (`fn is None`, i.e. driven by `variable`/`min`/`max`)
in `PI0_CUT_LISTS['1shw']` with genuinely finite bounds on both sides -- an
open-ended cut like `nu_score` (`min=0.5, max=inf`) would make a naive
`(min+max)/2` midpoint a no-op (`inf` either way), so those are skipped in
favor of one with real, finite bounds. Narrows it to the midpoint --
same name as what built the store, genuinely different selection.

In [ ]:
target_cut = None
for c in ana.PI0_CUT_LISTS['1shw']:
    if c.fn is not None:
        continue
    if np.isfinite(c.min) and np.isfinite(c.max):
        target_cut = c
        break
if target_cut is None:
    target_cut = next(c for c in ana.PI0_CUT_LISTS['1shw'] if c.fn is None)

if np.isfinite(target_cut.min) and np.isfinite(target_cut.max):
    new_min, new_max = target_cut.min, (target_cut.min + target_cut.max) / 2
elif np.isfinite(target_cut.min):
    new_min, new_max = target_cut.min + 1.0, target_cut.max
else:
    new_min, new_max = target_cut.min, target_cut.max - 1.0

print(f"Narrowing cut '{target_cut.name}': ({target_cut.min}, {target_cut.max}) -> ({new_min}, {new_max})")
drifted_cuts = ana.modify_cut(ana.PI0_CUT_LISTS['1shw'], target_cut.name, min=new_min, max=new_max)

try:
    ana.get_detvar_systs(hnl_detvar_1shw, var, bins, cuts=drifted_cuts)
    print("FAIL: expected a drift ValueError, nothing was raised")
except ValueError as e:
    if "drifted" in str(e):
        print("PASS: drift correctly detected and raised:")
        print(e)
    else:
        print("FAIL: raised, but not the drift error -- got:")
        print(e)

## Case (c): live cuts missing a stamped cut entirely -- should raise

In [ ]:
dropped_name = ana.PI0_CUT_LISTS['1shw'][0].name
incomplete_cuts = ana.drop_cuts(ana.PI0_CUT_LISTS['1shw'], dropped_name)
print(f"Dropped cut '{dropped_name}' from the live cuts list")

try:
    ana.get_detvar_systs(hnl_detvar_1shw, var, bins, cuts=incomplete_cuts)
    print("FAIL: expected a missing-cut ValueError, nothing was raised")
except ValueError as e:
    if "missing" in str(e):
        print("PASS: missing cut correctly detected and raised:")
        print(e)
    else:
        print("FAIL: raised, but not the missing-cut error -- got:")
        print(e)

## Case (d): live cuts legitimately extend the stamp (narrowing case) -- should run clean

In [ ]:
extra_cut = ana.CutSpec("extra_test_cut", fn=lambda df: np.ones(len(df), dtype=bool))  # always-true, just tests the code path
extended_cuts = ana.PI0_CUT_LISTS['1shw'] + [extra_cut]

try:
    ana.get_detvar_systs(hnl_detvar_1shw, var, bins, cuts=extended_cuts)
    print("PASS: legitimate narrowing ran clean, no error")
except ValueError as e:
    print("FAIL: unexpected raise on a cuts list that only adds to the stamp:")
    print(e)

## Summary

If all four cases above printed PASS, the drift check (Part B of the
`get_total_cov`/`SystematicsInput` redesign) is confirmed working end-to-end
against a real store for the first time.